# Tiền xử lý dữ liệu Loan Approval (archive.csv)

---

**Bộ dữ liệu:** 614 dòng, 13 cột — dữ liệu về các khoản vay cá nhân, mục tiêu dự đoán khoản vay có được duyệt hay không (`Loan_Status`).

**Các bước tiền xử lý:**
1. Đọc và khám phá dữ liệu
2. Xử lý giá trị thiếu
3. Xử lý outlier
4. Mã hóa biến phân loại
5. Chuẩn hóa dữ liệu số

In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

---
## 1. Đọc và khám phá dữ liệu

Xem tổng quan dữ liệu để xác định: kiểu dữ liệu, số lượng null, phân phối các cột.

In [18]:
df = pd.read_csv("archive.csv")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 83.4 KB


In [20]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [21]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

**Nhận xét:**
- 6 cột có giá trị thiếu: `Gender` (13), `Married` (3), `Dependents` (15), `Self_Employed` (32), `LoanAmount` (22), `Loan_Amount_Term` (14), `Credit_History` (50).
- `ApplicantIncome` và `LoanAmount` có độ lệch lớn (skew cao), khả năng có outlier.
- `Dependents` chứa giá trị `"3+"` cần chuyển về số.
- `Loan_ID` là mã định danh, không có ý nghĩa cho mô hình → bỏ.

In [22]:
df = df.drop(columns=["Loan_ID"])

---
## 2. Xử lý giá trị thiếu

**Lý do:** Mô hình ML không chấp nhận giá trị null. Cần điền giá trị phù hợp để giữ lại dữ liệu thay vì xóa dòng.

**Phương pháp:**
- Biến phân loại (`Gender`, `Married`, `Dependents`, `Self_Employed`): điền bằng **mode** (giá trị xuất hiện nhiều nhất) — giữ đúng phân phối gốc.
- `LoanAmount`: điền bằng **median** — không bị ảnh hưởng bởi outlier như mean.
- `Loan_Amount_Term`: điền bằng **mode** (360.0 chiếm đa số).
- `Credit_History`: điền bằng **mode** (1.0 chiếm 84%) — đây là biến nhị phân, mode phù hợp nhất.

In [23]:
for col in ["Gender", "Married", "Dependents", "Self_Employed", "Loan_Amount_Term", "Credit_History"]:
    df[col] = df[col].fillna(df[col].mode()[0])

df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

In [24]:
df.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

**Kết quả:** Tất cả cột đều có 0 giá trị null.

---
## 3. Xử lý outlier

**Lý do:** `ApplicantIncome`, `CoapplicantIncome`, `LoanAmount` có độ lệch rất cao (skewness lần lượt 6.5, 7.5, 2.7). Outlier gây sai lệch mô hình và ảnh hưởng chuẩn hóa.

**Phương pháp:** Biến đổi **log1p** — giảm skew và thu hẹp khoảng giá trị mà không mất dữ liệu. Dùng `log1p` (log(1+x)) thay vì `log` để xử lý giá trị 0 trong `CoapplicantIncome`.

In [25]:
skew_before = df[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]].skew()
print("Skewness TRƯỚC:")
print(skew_before)

for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
    df[col] = np.log1p(df[col])

skew_after = df[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]].skew()
print("\nSkewness SAU:")
print(skew_after)

Skewness TRƯỚC:
ApplicantIncome      6.539513
CoapplicantIncome    7.491531
LoanAmount           2.743053
dtype: float64

Skewness SAU:
ApplicantIncome      0.482128
CoapplicantIncome   -0.173073
LoanAmount          -0.151578
dtype: float64


**Kết quả:** Skewness giảm đáng kể (từ 6–7 về gần 0–1), phân phối đối xứng hơn.

---
## 4. Mã hóa biến phân loại

**Lý do:** Mô hình ML chỉ nhận giá trị số, cần chuyển chuỗi về số.

**Phương pháp:**
- Biến nhị phân (`Gender`, `Married`, `Education`, `Self_Employed`, `Loan_Status`): **Label Encoding** (0/1) — đơn giản và phù hợp với 2 giá trị.
- `Dependents`: chuyển `"3+"` → `3`, sau đó ép kiểu `int` — giữ đúng thứ tự số.
- `Property_Area` (3 giá trị, không có thứ tự): **One-Hot Encoding** — tránh mô hình hiểu sai là có quan hệ lớn nhỏ giữa các vùng.

In [26]:
df["Dependents"] = df["Dependents"].replace("3+", "3").astype(int)

le = LabelEncoder()
for col in ["Gender", "Married", "Education", "Self_Employed", "Loan_Status"]:
    df[col] = le.fit_transform(df[col])

df = pd.get_dummies(df, columns=["Property_Area"], drop_first=True, dtype=int)

In [27]:
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,0,0,8.674197,0.000000,4.859812,360.0,1.0,1,0,1
1,1,1,1,0,0,8.430327,7.319202,4.859812,360.0,1.0,0,0,0
2,1,1,0,0,1,8.006701,0.000000,4.204693,360.0,1.0,1,0,1
3,1,1,0,1,0,7.857094,7.765993,4.795791,360.0,1.0,1,0,1
4,1,0,0,0,0,8.699681,0.000000,4.955827,360.0,1.0,1,0,1


**Kết quả:** Tất cả cột đều là kiểu số. `Property_Area` được tách thành 2 cột dummy (`Semiurban`, `Urban`), bỏ `Rural` làm baseline.

---
## 5. Chuẩn hóa dữ liệu số

**Lý do:** Các cột số có khoảng giá trị khác nhau (ví dụ: income hàng nghìn vs. Credit_History 0–1). Chuẩn hóa giúp mô hình hội tụ nhanh và không ưu tiên cột có giá trị lớn.

**Phương pháp:** `StandardScaler` (z-score: mean=0, std=1) — phù hợp với dữ liệu đã được giảm skew. Chỉ áp dụng cho cột số liên tục, không scale cột nhị phân/dummy.

In [28]:
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [29]:
df.describe()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
count,614.000000,614.000000,614.000000,614.000000,614.000000,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,614.000000,614.000000,614.000000,614.000000
mean,0.817590,0.653094,0.744300,0.218241,0.133550,9.453169e-16,-2.893089e-18,3.761016e-16,5.930833e-17,0.855049,0.687296,0.379479,0.328990
std,0.386497,0.476373,1.009623,0.413389,0.340446,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,0.352339,0.463973,0.485653,0.470229
min,0.000000,0.000000,0.000000,0.000000,0.000000,-5.157770e+00,-1.107783e+00,-5.227264e+00,-5.132498e+00,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,-5.841435e-01,-1.107783e+00,-5.067337e-01,2.732313e-01,1.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,0.000000,0.000000,0.000000,-1.477208e-01,7.206820e-01,-1.280320e-02,2.732313e-01,1.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,0.000000,0.000000,5.017959e-01,8.907879e-01,4.983292e-01,2.732313e-01,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,3.000000,1.000000,1.000000,4.593738e+00,1.638995e+00,3.438784e+00,2.137276e+00,1.000000,1.000000,1.000000,1.000000


**Kết quả:** Các cột số liên tục có mean ≈ 0 và std ≈ 1.

---
## Tổng kết

| Bước | Phương pháp | Lý do |
|-------|------------|-------|
| Xử lý null | Mode (phân loại), Median (số) | Giữ phân phối gốc, không bị outlier ảnh hưởng |
| Xử lý outlier | Log1p transform | Giảm skew, giữ nguyên dữ liệu |
| Mã hóa | Label Encoding (nhị phân), One-Hot (nhiều giá trị) | Chuyển chuỗi → số, tránh ordinal sai |
| Chuẩn hóa | StandardScaler | Cân bằng scale giữa các feature |

In [30]:
print(f"Shape sau tiền xử lý: {df.shape}")
print(f"Các cột: {list(df.columns)}")
df.head()

Shape sau tiền xử lý: (614, 13)
Các cột: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'Property_Area_Semiurban', 'Property_Area_Urban']


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,0,0,0.516186,-1.107783,-0.012803,0.273231,1.0,1,0,1
1,1,1,1,0,0,0.137806,0.782158,-0.012803,0.273231,1.0,0,0,0
2,1,1,0,0,1,-0.519479,-1.107783,-1.348663,0.273231,1.0,1,0,1
3,1,1,0,1,0,-0.751605,0.897526,-0.143351,0.273231,1.0,1,0,1
4,1,0,0,0,0,0.555727,-1.107783,0.182981,0.273231,1.0,1,0,1


In [31]:
df.to_csv("data/loan_preprocessed.csv", index=False)
print("Đã lưu file: data/loan_preprocessed.csv")

Đã lưu file: data/loan_preprocessed.csv
